# 🧠 DeepSeek MoE Architecture — PyTorch Tutorial from Scratch

We'll build the **full DeepSeek-MoE architecture** piece by piece, explaining every component.

### What we'll build:
1. 🔧 PyTorch Basics Refresher
2. 📐 RMSNorm
3. 🔄 RoPE (Rotary Position Embeddings)
4. 👁️ Self-Attention (Q, K, V, O projections)
5. 🚪 SwiGLU FFN (gate / up / down projections)
6. 🧑‍🤝‍🧑 MoE Layer (routing + experts)
7. 🏗️ Full Mini DeepSeek Model
8. 🔬 Visualizing Expert Routing


In [ ]:
# Install dependencies if needed
# !pip install torch matplotlib seaborn

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import math
import numpy as np

# Reproducibility
torch.manual_seed(42)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')

---
## 🔧 Part 1: PyTorch Basics Refresher

Before we dive in, a quick refresher on the core PyTorch building blocks we'll use throughout.

In [ ]:
# ─────────────────────────────────────────────
# 1a. Tensors — the fundamental data structure
# ─────────────────────────────────────────────
#
# Think of tensors as multi-dimensional arrays.
# In NLP, we usually work with shape: (batch, seq_len, hidden_dim)
#   batch     = how many sentences at once
#   seq_len   = number of tokens in the sentence
#   hidden_dim= the vector size for each token

batch_size = 2
seq_len    = 5
hidden_dim = 8

x = torch.randn(batch_size, seq_len, hidden_dim)
print(f'Input shape: {x.shape}')   # (2, 5, 8)
print(f'  batch_size={batch_size}, seq_len={seq_len}, hidden_dim={hidden_dim}')

In [ ]:
# ─────────────────────────────────────────────
# 1b. nn.Linear — the core projection layer
# ─────────────────────────────────────────────
#
# nn.Linear(in, out) multiplies input by a weight matrix W
# output = x @ W.T   (matrix multiply)
# This is how ALL projections (Q, K, V, gate, up, down) work.

linear = nn.Linear(in_features=8, out_features=16, bias=False)
out = linear(x)  # x: (2,5,8) → out: (2,5,16)
print(f'After linear(8→16): {out.shape}')
print(f'Weight matrix shape: {linear.weight.shape}')  # (16, 8) — stores 16×8 = 128 parameters

In [ ]:
# ─────────────────────────────────────────────
# 1c. nn.Module — how we build custom layers
# ─────────────────────────────────────────────
#
# Every component we build inherits from nn.Module.
# You define:
#   __init__  : set up sub-layers and parameters
#   forward   : define how data flows through

class SimpleLayer(nn.Module):
    def __init__(self, dim):
        super().__init__()              # always call this first
        self.linear = nn.Linear(dim, dim)
        self.act    = nn.SiLU()         # SiLU activation (used by DeepSeek)

    def forward(self, x):
        return self.act(self.linear(x)) # data flows: x → linear → SiLU → output

layer = SimpleLayer(dim=8)
out = layer(x)
print(f'SimpleLayer output shape: {out.shape}')  # same shape as input

In [ ]:
# ─────────────────────────────────────────────
# 1d. Quick look at activations
# ─────────────────────────────────────────────
#
# Activations add non-linearity — without them, stacking
# linear layers is the same as one linear layer.

x_plot = torch.linspace(-4, 4, 100)

plt.figure(figsize=(10, 3))
plt.plot(x_plot, F.relu(x_plot),  label='ReLU  (old)')
plt.plot(x_plot, F.gelu(x_plot),  label='GELU  (GPT-2/BERT)')
plt.plot(x_plot, F.silu(x_plot),  label='SiLU  (DeepSeek ← this one!)', linewidth=2.5)
plt.axhline(0, color='gray', linewidth=0.5)
plt.axvline(0, color='gray', linewidth=0.5)
plt.title('Activation Functions')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print('SiLU(x) = x * sigmoid(x) — smooth, allows small negative values')

---
## 📐 Part 2: RMSNorm

Before each major block (attention and FFN), DeepSeek applies **RMSNorm** to stabilize training.

It normalizes the vector so its magnitude doesn't explode or vanish as it passes through many layers.

$$\text{RMSNorm}(x) = \frac{x}{\text{RMS}(x)} \cdot \gamma \qquad \text{where} \quad \text{RMS}(x) = \sqrt{\frac{1}{d}\sum_i x_i^2}$$

Simpler than LayerNorm — no mean subtraction, just scale by the root-mean-square.

In [ ]:
class RMSNorm(nn.Module):
    """
    Root Mean Square Layer Normalization.
    
    Args:
        dim: the hidden dimension size to normalize over
        eps: small value to prevent division by zero
    """
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        # γ (gamma) is a learnable scale parameter, initialized to ones
        # The model learns how much to scale after normalization
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Step 1: Compute RMS along the last dimension (hidden_dim)
        #   x.pow(2)       → square every element
        #   .mean(-1, keepdim=True) → average across hidden_dim
        #   .rsqrt()       → 1 / sqrt(mean)
        rms_inv = x.pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt()
        
        # Step 2: Normalize x, then multiply by learned scale γ
        return x * rms_inv * self.weight

# ── Test it ──────────────────────────────────────────
norm = RMSNorm(dim=hidden_dim)
x_test = torch.randn(2, 5, hidden_dim) * 100  # large values
x_normed = norm(x_test)

print(f'Before norm — mean: {x_test.mean():.2f}, std: {x_test.std():.2f}')
print(f'After norm  — mean: {x_normed.mean():.4f}, std: {x_normed.std():.4f}')
print(f'\nShape unchanged: {x_test.shape} → {x_normed.shape}')

---
## 🔄 Part 3: RoPE — Rotary Position Embeddings

Attention by itself has **no sense of position** — token 1 and token 100 look the same.

RoPE injects position information by **rotating** the Q and K vectors before computing attention scores. Tokens that are far apart get rotated more, so their dot product naturally decreases.

> 💡 Intuition: imagine two clock hands. Hands far apart in time point in very different directions, so their dot product (cosine similarity) is small.

In [ ]:
class RotaryEmbedding(nn.Module):
    """
    Rotary Position Embedding (RoPE).
    Pre-computes sin/cos values for all positions up to max_seq_len.
    """
    def __init__(self, dim: int, max_seq_len: int = 2048, base: float = 10000.0):
        super().__init__()
        # Compute θ_i = 1 / (base^(2i/dim)) for i in 0..dim/2
        # These are the rotation frequencies — lower dims rotate faster
        theta = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer('theta', theta)  # save as non-parameter buffer

        # Pre-compute for all positions [0, 1, 2, ..., max_seq_len-1]
        positions = torch.arange(max_seq_len).float()
        # Outer product: each position × each frequency
        freqs = torch.outer(positions, theta)           # (max_seq_len, dim/2)
        # Stack [freqs, freqs] so it matches full dim
        emb = torch.cat([freqs, freqs], dim=-1)         # (max_seq_len, dim)
        self.register_buffer('cos_cached', emb.cos())
        self.register_buffer('sin_cached', emb.sin())

    def forward(self, x: torch.Tensor, seq_len: int):
        """Returns (cos, sin) for positions 0..seq_len-1"""
        return (
            self.cos_cached[:seq_len].unsqueeze(0).unsqueeze(0),  # (1,1,seq,dim)
            self.sin_cached[:seq_len].unsqueeze(0).unsqueeze(0),
        )

def rotate_half(x: torch.Tensor) -> torch.Tensor:
    """Rotate x by swapping and negating the two halves — the 2D rotation trick."""
    x1 = x[..., : x.shape[-1] // 2]   # first half
    x2 = x[..., x.shape[-1] // 2 :]   # second half
    return torch.cat([-x2, x1], dim=-1)

def apply_rope(q: torch.Tensor, k: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor):
    """
    Apply rotary embeddings to Q and K.
    The rotation: v_rotated = v * cos + rotate_half(v) * sin
    """
    q_rot = q * cos + rotate_half(q) * sin
    k_rot = k * cos + rotate_half(k) * sin
    return q_rot, k_rot

# ── Visualize RoPE ───────────────────────────────────────
rope = RotaryEmbedding(dim=64)
cos, sin = rope(None, seq_len=20)

plt.figure(figsize=(12, 3))
plt.subplot(1,2,1)
plt.imshow(cos[0,0].numpy(), aspect='auto', cmap='RdBu')
plt.title('RoPE cosines (20 positions × 64 dims)')
plt.xlabel('Dimension'); plt.ylabel('Position')
plt.colorbar()

plt.subplot(1,2,2)
# Show how similarity decreases with distance for a fixed query
q_vec = torch.randn(1, 1, 1, 64)
cos_all, sin_all = rope(None, seq_len=50)
sims = []
q_rot0, _ = apply_rope(q_vec, q_vec, cos_all[:,:,:1,:], sin_all[:,:,:1,:])
for pos in range(50):
    q_rot_i, _ = apply_rope(q_vec, q_vec, cos_all[:,:,pos:pos+1,:], sin_all[:,:,pos:pos+1,:])
    sim = F.cosine_similarity(q_rot0.flatten(), q_rot_i.flatten(), dim=0).item()
    sims.append(sim)
plt.plot(sims)
plt.title('RoPE: Q similarity vs distance from position 0')
plt.xlabel('Position offset'); plt.ylabel('Cosine similarity')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print('Tokens further apart → lower similarity (attention naturally decays with distance)')

---
## 👁️ Part 4: Self-Attention

The attention mechanism lets each token **look at all other tokens** and decide what to focus on.

```
Q = "What am I looking for?"
K = "What do I offer?"
V = "What do I actually pass forward?"
```

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

In [ ]:
class DeepseekAttention(nn.Module):
    """
    Multi-head Self-Attention with RoPE.
    
    Args:
        hidden_dim : size of the token embedding vector (e.g. 2048)
        num_heads  : number of parallel attention heads (e.g. 16)
    """
    def __init__(self, hidden_dim: int, num_heads: int):
        super().__init__()
        self.hidden_dim  = hidden_dim
        self.num_heads   = num_heads
        self.head_dim    = hidden_dim // num_heads  # each head gets a slice

        # ── The 4 projections ───────────────────────────────────────
        # All map hidden_dim → hidden_dim, but conceptually different:
        self.q_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)  # queries
        self.k_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)  # keys
        self.v_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)  # values
        self.o_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)  # output mix

        # RoPE — shared rotary embeddings for position encoding
        self.rotary_emb = RotaryEmbedding(dim=self.head_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape  # (batch, seq_len, hidden_dim)
        H = self.num_heads
        D = self.head_dim

        # ── Step 1: Project input into Q, K, V ───────────────────────
        # Each token gets its own query, key, value vector
        q = self.q_proj(x)  # (B, T, hidden_dim)
        k = self.k_proj(x)  # (B, T, hidden_dim)
        v = self.v_proj(x)  # (B, T, hidden_dim)

        # ── Step 2: Split into multiple heads ─────────────────────────
        # Reshape from (B, T, hidden_dim) → (B, H, T, head_dim)
        # Each head operates on a smaller slice of the embedding
        q = q.view(B, T, H, D).transpose(1, 2)  # (B, H, T, D)
        k = k.view(B, T, H, D).transpose(1, 2)
        v = v.view(B, T, H, D).transpose(1, 2)

        # ── Step 3: Apply RoPE to Q and K ────────────────────────────
        cos, sin = self.rotary_emb(x, seq_len=T)
        q, k = apply_rope(q, k, cos, sin)

        # ── Step 4: Compute attention scores ─────────────────────────
        # scores[b, h, i, j] = how much token i attends to token j
        scale  = math.sqrt(D)
        scores = torch.matmul(q, k.transpose(-2, -1)) / scale  # (B, H, T, T)

        # Causal mask: token i can only attend to tokens 0..i (no future peeking)
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        scores = scores.masked_fill(mask, float('-inf'))

        # Softmax → attention weights (sum to 1 over the attended-to tokens)
        attn_weights = F.softmax(scores, dim=-1)  # (B, H, T, T)

        # ── Step 5: Weighted sum of values ───────────────────────────
        out = torch.matmul(attn_weights, v)  # (B, H, T, D)

        # ── Step 6: Merge heads back + output projection ─────────────
        out = out.transpose(1, 2).contiguous().view(B, T, C)  # (B, T, hidden_dim)
        out = self.o_proj(out)

        return out, attn_weights  # return weights for visualization

# ── Test ────────────────────────────────────────────────
attn = DeepseekAttention(hidden_dim=64, num_heads=4)
x_test = torch.randn(2, 10, 64)  # batch=2, seq=10, dim=64
out, weights = attn(x_test)
print(f'Input shape  : {x_test.shape}')
print(f'Output shape : {out.shape}')      # same as input — attention is shape-preserving
print(f'Attn weights : {weights.shape}')  # (batch, heads, seq, seq)

In [ ]:
# ── Visualize attention weights ──────────────────────────────────────────────
# This shows which tokens each token is attending to

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
tokens = [f'T{i}' for i in range(10)]

for head_idx, ax in enumerate(axes):
    # Take batch item 0, head head_idx
    w = weights[0, head_idx].detach().numpy()
    sns.heatmap(w, ax=ax, cmap='Blues', xticklabels=tokens, yticklabels=tokens,
                vmin=0, vmax=1, cbar=(head_idx==3))
    ax.set_title(f'Head {head_idx+1}')
    ax.set_xlabel('Key (attended to)')
    if head_idx == 0:
        ax.set_ylabel('Query (attending from)')

plt.suptitle('Attention weights per head (lower triangle = causal mask)', y=1.02)
plt.tight_layout()
plt.show()
print('Each row sums to 1.0 — it\'s a distribution over which past tokens to attend to')

---
## 🚪 Part 5: SwiGLU FFN (gate / up / down projections)

After attention, each token passes through a feedforward network **independently**.

DeepSeek uses **SwiGLU**, which needs 3 projections:

```
         ┌── gate_proj ──→ SiLU ──┐
x ───────┤                        ⊗ (multiply) ──→ down_proj ──→ output
         └── up_proj ─────────────┘
```

The **gate** acts as a learned on/off switch for each dimension of the **up** projection.

In [ ]:
class SwiGLU_FFN(nn.Module):
    """
    SwiGLU Feed-Forward Network.
    
    This is used as the FFN in Layer 0 (the dense layer) and
    in each individual expert in the MoE layers.
    
    Args:
        hidden_dim  : input/output dimension (e.g. 2048)
        ffn_dim     : inner dimension — larger = more capacity
    """
    def __init__(self, hidden_dim: int, ffn_dim: int):
        super().__init__()

        # These two projections expand the input to ffn_dim
        self.gate_proj = nn.Linear(hidden_dim, ffn_dim, bias=False)
        self.up_proj   = nn.Linear(hidden_dim, ffn_dim, bias=False)

        # This projection squishes back down to hidden_dim
        self.down_proj = nn.Linear(ffn_dim, hidden_dim, bias=False)

        self.act_fn = nn.SiLU()  # Sigmoid Linear Unit

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # ── The SwiGLU computation ────────────────────────────────
        gate = self.act_fn(self.gate_proj(x))  # SiLU acts as a soft gate: values
                                                # near 0 get squashed, large values pass
        up   = self.up_proj(x)                 # the "content" to be gated
        
        # Element-wise multiply: gate decides how much of each 'up' dimension flows
        hidden = gate * up                     # (B, T, ffn_dim)

        # Project back to model dimension
        return self.down_proj(hidden)          # (B, T, hidden_dim)

# ── Test & visualize ──────────────────────────────────────────────────────────
ffn = SwiGLU_FFN(hidden_dim=64, ffn_dim=256)
x_test = torch.randn(2, 10, 64)
out = ffn(x_test)
print(f'FFN input  shape: {x_test.shape}')
print(f'FFN output shape: {out.shape}')  # same shape as input

# Show the gating mechanism in action
x_single = torch.randn(1, 1, 64)
with torch.no_grad():
    gate_vals = torch.sigmoid(ffn.gate_proj(x_single)).squeeze()   # gate values 0→1
    up_vals   = ffn.up_proj(x_single).squeeze()

plt.figure(figsize=(12, 3))
plt.subplot(1,3,1)
plt.bar(range(min(50, 256)), gate_vals[:50].detach().numpy(), color='coral')
plt.title('Gate values (first 50 dims)\n→ decides what passes through')
plt.ylim(0,1); plt.xlabel('FFN dimension')

plt.subplot(1,3,2)
plt.bar(range(min(50, 256)), up_vals[:50].detach().numpy(), color='steelblue')
plt.title('Up values (first 50 dims)\n→ the content to be filtered')
plt.xlabel('FFN dimension')

plt.subplot(1,3,3)
gated = (gate_vals * up_vals)[:50].detach().numpy()
plt.bar(range(min(50, 256)), gated, color='green')
plt.title('Gate × Up (first 50 dims)\n→ filtered content')
plt.xlabel('FFN dimension')

plt.tight_layout()
plt.show()

---
## 🧑‍🤝‍🧑 Part 6: MoE Layer — The Heart of DeepSeek

Instead of **one big FFN** for all tokens, MoE has:
- **64 small expert FFNs** (each specialized)
- **1 always-on shared expert** (handles common knowledge)
- **A gate router** that picks the top-K experts per token

Key insight: each token only uses 6 experts out of 64, so **compute is sparse** but **capacity is huge**.

In [ ]:
class MoEGate(nn.Module):
    """
    The routing gate — decides which experts each token goes to.
    
    For each token, it:
    1. Computes a score for each expert (linear projection)
    2. Takes top-K experts by score
    3. Softmax over only those top-K scores → routing weights
    """
    def __init__(self, hidden_dim: int, num_experts: int, top_k: int):
        super().__init__()
        self.num_experts = num_experts
        self.top_k       = top_k
        # A simple linear layer: hidden_dim → num_experts scores
        self.weight = nn.Linear(hidden_dim, num_experts, bias=False)

    def forward(self, x: torch.Tensor):
        """
        Args:
            x: (B, T, hidden_dim) token representations
        Returns:
            routing_weights: (B*T, top_k) — how much weight for each chosen expert
            selected_experts:(B*T, top_k) — which experts were chosen (indices)
        """
        B, T, C = x.shape
        # Flatten batch and sequence: treat every token independently
        x_flat = x.view(B * T, C)                          # (B*T, hidden_dim)

        # Compute raw scores for every expert
        logits = self.weight(x_flat)                       # (B*T, num_experts)

        # Pick the top-K experts for each token
        topk_scores, topk_indices = torch.topk(logits, self.top_k, dim=-1)
        #   topk_scores:  (B*T, top_k) — raw scores of chosen experts
        #   topk_indices: (B*T, top_k) — which expert indices were chosen

        # Softmax only over the top-K (normalize so weights sum to 1)
        routing_weights = F.softmax(topk_scores, dim=-1)   # (B*T, top_k)

        return routing_weights, topk_indices, logits        # return logits for aux loss


class DeepseekMoE(nn.Module):
    """
    DeepSeek MoE Layer.
    
    Key design choices vs standard MoE:
    - Fine-grained: more (64) but smaller experts
    - Shared expert: always-on expert for common computations
    
    Args:
        hidden_dim    : token embedding size (e.g. 2048)
        num_experts   : number of routed experts (e.g. 64)
        expert_ffn_dim: inner dim of each expert FFN (e.g. 1408)
        shared_ffn_dim: inner dim of shared expert (e.g. 2816 — 2× expert)
        top_k         : how many experts each token uses (e.g. 6)
    """
    def __init__(
        self,
        hidden_dim: int,
        num_experts: int,
        expert_ffn_dim: int,
        shared_ffn_dim: int,
        top_k: int,
    ):
        super().__init__()
        self.hidden_dim  = hidden_dim
        self.num_experts = num_experts
        self.top_k       = top_k

        # ── 64 routed experts (small, specialized) ──────────────────
        # ModuleList so PyTorch tracks all their parameters
        self.experts = nn.ModuleList([
            SwiGLU_FFN(hidden_dim=hidden_dim, ffn_dim=expert_ffn_dim)
            for _ in range(num_experts)
        ])

        # ── 1 shared expert (larger, always active) ─────────────────
        self.shared_expert = SwiGLU_FFN(hidden_dim=hidden_dim, ffn_dim=shared_ffn_dim)

        # ── Router / gate ────────────────────────────────────────────
        self.gate = MoEGate(hidden_dim=hidden_dim, num_experts=num_experts, top_k=top_k)

    def forward(self, x: torch.Tensor):
        B, T, C = x.shape
        x_flat = x.view(B * T, C)   # (B*T, hidden_dim)

        # ── Step 1: Routing ──────────────────────────────────────────
        routing_weights, selected_experts, gate_logits = self.gate(x)
        # routing_weights:   (B*T, top_k)  — how much each chosen expert contributes
        # selected_experts:  (B*T, top_k)  — which expert index was chosen

        # ── Step 2: Shared expert (always runs, no routing needed) ───
        shared_out = self.shared_expert(x_flat)  # (B*T, hidden_dim)

        # ── Step 3: Routed experts (sparse computation) ──────────────
        routed_out = torch.zeros_like(x_flat)    # accumulator

        for expert_idx in range(self.num_experts):
            # Find which tokens chose this expert
            # selected_experts: (B*T, top_k) — check if expert_idx appears
            expert_mask = (selected_experts == expert_idx)  # (B*T, top_k) bool

            # Get (token_id, position_in_topk) pairs where this expert was selected
            token_ids, topk_pos = expert_mask.nonzero(as_tuple=True)

            if len(token_ids) == 0:
                continue  # this expert wasn't chosen by any token — skip!

            # Get the tokens that go to this expert
            tokens_for_expert = x_flat[token_ids]            # (n_tokens, hidden_dim)

            # Run the expert FFN on those tokens
            expert_out = self.experts[expert_idx](tokens_for_expert)  # (n_tokens, hidden_dim)

            # Weight by routing score and accumulate
            weights = routing_weights[token_ids, topk_pos].unsqueeze(-1)  # (n_tokens, 1)
            routed_out.index_add_(0, token_ids, weights * expert_out)

        # ── Step 4: Combine shared + routed outputs ──────────────────
        output = shared_out + routed_out          # (B*T, hidden_dim)
        output = output.view(B, T, C)             # restore (B, T, hidden_dim)

        return output, gate_logits


# ── Test ────────────────────────────────────────────────────────────────────
moe_layer = DeepseekMoE(
    hidden_dim=64,
    num_experts=8,       # using 8 for speed (real model has 64)
    expert_ffn_dim=128,  # each expert's inner dim
    shared_ffn_dim=256,  # shared expert inner dim (2×)
    top_k=2,             # each token uses top 2 experts
)

x_test = torch.randn(2, 10, 64)
out, logits = moe_layer(x_test)
print(f'MoE input  shape: {x_test.shape}')
print(f'MoE output shape: {out.shape}')       # unchanged
print(f'Gate logits shape: {logits.shape}')   # (B*T, num_experts)

---
## 🔬 Part 7: Auxiliary Loss (Expert Specialization)

A well-known problem in MoE training is **load imbalance**: the router keeps picking the same few experts, and the rest never get trained.

The paper "Auxloss For Advancing Expert Specialization" addresses this. The most common fix is a **load balancing auxiliary loss** that penalizes unequal expert utilization.

$$\mathcal{L}_{aux} = \alpha \sum_{i=1}^{N} f_i \cdot P_i$$

Where $f_i$ = fraction of tokens routed to expert $i$, and $P_i$ = average gate probability for expert $i$.

In [ ]:
def auxiliary_load_balance_loss(
    gate_logits: torch.Tensor,
    selected_experts: torch.Tensor,
    num_experts: int,
    alpha: float = 0.01
) -> torch.Tensor:
    """
    Auxiliary loss to encourage equal expert utilization.
    
    If all tokens go to the same expert: loss is HIGH → penalized
    If tokens are spread evenly across experts: loss is LOW → rewarded
    
    Args:
        gate_logits      : (num_tokens, num_experts) — raw router scores
        selected_experts : (num_tokens, top_k) — which experts were chosen
        num_experts      : total number of experts
        alpha            : weight of this loss relative to main LM loss
    """
    num_tokens = gate_logits.shape[0]

    # f_i: fraction of tokens routed to expert i (hard count)
    # Count how many times each expert appears in selected_experts
    expert_counts = torch.zeros(num_experts, device=gate_logits.device)
    for i in range(num_experts):
        expert_counts[i] = (selected_experts == i).sum().float()
    f = expert_counts / num_tokens  # normalize to fraction

    # P_i: mean gate probability for expert i (soft, differentiable)
    gate_probs = F.softmax(gate_logits, dim=-1)  # (num_tokens, num_experts)
    P = gate_probs.mean(dim=0)                   # (num_experts,)

    # Loss = sum of f_i * P_i, scaled by num_experts (so perfect balance = 1.0)
    # This product is minimized when both f and P are uniform
    loss = num_experts * (f * P).sum()

    return alpha * loss


# ── Visualize: what happens with balanced vs unbalanced routing ──────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, scenario in zip(axes, ['Unbalanced (bad)', 'Balanced (good)']):
    if 'Unbalanced' in scenario:
        # Most tokens go to expert 0 — router collapsed!
        logits = torch.randn(100, 8) * 0.1
        logits[:, 0] += 3.0  # expert 0 always wins
    else:
        # Tokens spread evenly
        logits = torch.randn(100, 8)

    _, top2 = torch.topk(logits, 2, dim=-1)
    counts = [(top2 == i).sum().item() for i in range(8)]

    loss_val = auxiliary_load_balance_loss(logits, top2, num_experts=8).item()

    ax.bar(range(8), counts, color=['red' if c > 50 else 'steelblue' for c in counts])
    ax.axhline(100*2/8, color='green', linestyle='--', label=f'Ideal (uniform = {100*2//8})')
    ax.set_title(f'{scenario}\nAux loss = {loss_val:.4f}')
    ax.set_xlabel('Expert ID')
    ax.set_ylabel('Number of tokens assigned')
    ax.legend()

plt.suptitle('Expert Load Balancing', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🏗️ Part 8: Full Mini DeepSeek Model

Now we put it all together into a complete (but small) DeepSeek-style model.

In [ ]:
class DeepseekDecoderLayer(nn.Module):
    """
    One full transformer block:
        RMSNorm → Attention → residual
        RMSNorm → FFN/MoE  → residual

    Args:
        layer_idx : 0 uses dense FFN, >0 uses MoE
        config    : dict of model hyperparameters
    """
    def __init__(self, layer_idx: int, config: dict):
        super().__init__()
        self.layer_idx = layer_idx
        C = config['hidden_dim']

        # Pre-norm for attention
        self.input_layernorm = RMSNorm(C)
        # Attention block
        self.self_attn = DeepseekAttention(
            hidden_dim=C,
            num_heads=config['num_heads']
        )
        # Pre-norm for FFN
        self.post_attention_layernorm = RMSNorm(C)

        # FFN: dense for layer 0, MoE for all others
        if layer_idx == 0:
            self.mlp = SwiGLU_FFN(
                hidden_dim=C,
                ffn_dim=config['dense_ffn_dim']
            )
        else:
            self.mlp = DeepseekMoE(
                hidden_dim=C,
                num_experts=config['num_experts'],
                expert_ffn_dim=config['expert_ffn_dim'],
                shared_ffn_dim=config['shared_ffn_dim'],
                top_k=config['top_k']
            )

    def forward(self, x: torch.Tensor):
        gate_logits = None

        # ── Attention block (with residual connection) ───────────────
        # Residual: add input to output → prevents gradient vanishing
        residual = x
        x = self.input_layernorm(x)          # normalize first
        x, _ = self.self_attn(x)             # run attention
        x = residual + x                     # add skip connection

        # ── FFN/MoE block (with residual connection) ─────────────────
        residual = x
        x = self.post_attention_layernorm(x) # normalize first
        if self.layer_idx == 0:
            x = self.mlp(x)                  # dense FFN
        else:
            x, gate_logits = self.mlp(x)     # MoE
        x = residual + x                     # add skip connection

        return x, gate_logits


class MiniDeepSeek(nn.Module):
    """
    A small but architecturally faithful DeepSeek-MoE model.
    
    Real model config for reference:
      hidden_dim=2048, num_layers=28, num_heads=16,
      num_experts=64, expert_ffn_dim=1408,
      shared_ffn_dim=2816, top_k=6
    """
    def __init__(self, config: dict):
        super().__init__()
        self.config = config

        # Token embedding: vocab_size → hidden_dim
        self.embed_tokens = nn.Embedding(config['vocab_size'], config['hidden_dim'])

        # Stack of decoder layers
        self.layers = nn.ModuleList([
            DeepseekDecoderLayer(layer_idx=i, config=config)
            for i in range(config['num_layers'])
        ])

        # Final normalization before output
        self.norm = RMSNorm(config['hidden_dim'])

        # Language model head: hidden_dim → vocab_size (logit per token)
        self.lm_head = nn.Linear(config['hidden_dim'], config['vocab_size'], bias=False)

    def forward(self, input_ids: torch.Tensor):
        """
        Args:
            input_ids: (B, T) — token IDs
        Returns:
            logits: (B, T, vocab_size) — predicted next-token distribution
        """
        # Convert token IDs → dense vectors
        x = self.embed_tokens(input_ids)   # (B, T, hidden_dim)

        all_gate_logits = []
        for layer in self.layers:
            x, gate_logits = layer(x)
            if gate_logits is not None:
                all_gate_logits.append(gate_logits)

        x = self.norm(x)                   # final norm
        logits = self.lm_head(x)           # (B, T, vocab_size)

        return logits, all_gate_logits


# ── Mini config (reduced for fast testing) ──────────────────────────────────
config = {
    'vocab_size'    : 1000,
    'hidden_dim'    : 64,
    'num_layers'    : 4,    # 1 dense + 3 MoE
    'num_heads'     : 4,
    'dense_ffn_dim' : 256,  # layer 0 FFN inner dim
    'num_experts'   : 8,    # routed experts per MoE layer
    'expert_ffn_dim': 128,  # each expert's inner dim
    'shared_ffn_dim': 256,  # shared expert inner dim
    'top_k'         : 2,    # how many experts per token
}

model = MiniDeepSeek(config)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters    : {total_params:,}')
print(f'Trainable parameters: {trainable:,}')

# Forward pass
input_ids = torch.randint(0, config['vocab_size'], (2, 10))  # batch=2, seq=10
logits, all_gate_logits = model(input_ids)
print(f'\nInput shape    : {input_ids.shape}')
print(f'Logits shape   : {logits.shape}')             # (2, 10, 1000)
print(f'Gate logits from {len(all_gate_logits)} MoE layers')

In [ ]:
# ── Visualize full model architecture ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 10))
ax.axis('off')

components = [
    ('Token Embeddings\n(vocab→hidden_dim)', '#AED6F1'),
    ('─── Layer 0 (Dense) ───', '#F8F9FA'),
    ('  RMSNorm + Self-Attention\n  (Q,K,V,O projections + RoPE)', '#D5E8D4'),
    ('  RMSNorm + Dense FFN\n  (gate·up→down projections)', '#FFF2CC'),
    ('─── Layers 1-3 (MoE) ───', '#F8F9FA'),
    ('  RMSNorm + Self-Attention\n  (Q,K,V,O projections + RoPE)', '#D5E8D4'),
    ('  RMSNorm + MoE Layer\n  Gate → top-K of 8 experts\n  + always-on shared expert', '#FFE6CC'),
    ('  ↑ Repeated × 3', '#F8F9FA'),
    ('Final RMSNorm', '#AED6F1'),
    ('LM Head\n(hidden_dim → vocab_size logits)', '#AED6F1'),
]

y = 0.97
for text, color in components:
    height = 0.07 if '\n' in text else 0.04
    fancy = dict(boxstyle='round,pad=0.5', facecolor=color, edgecolor='#999', linewidth=0.8)
    ax.text(0.5, y, text, transform=ax.transAxes, fontsize=10,
            verticalalignment='top', horizontalalignment='center',
            bbox=fancy if color != '#F8F9FA' else None,
            color='#333' if color != '#F8F9FA' else '#666',
            fontstyle='italic' if color == '#F8F9FA' else 'normal')
    y -= height + 0.01
    if color != '#F8F9FA' and y > 0.05:
        ax.annotate('', xy=(0.5, y+0.01), xytext=(0.5, y+0.025),
                    arrowprops=dict(arrowstyle='->', color='#555'))

ax.set_title('MiniDeepSeek Architecture', fontsize=13, fontweight='bold', pad=10)
plt.tight_layout()
plt.show()

---
## 🔬 Part 9: Visualizing Expert Routing

One of the most interesting things about MoE is seeing **which experts handle which tokens**. Let's visualize the routing patterns.

In [ ]:
# Run the model and capture gate logits
batch_size_viz = 4
seq_len_viz    = 12

input_ids_viz = torch.randint(0, config['vocab_size'], (batch_size_viz, seq_len_viz))
with torch.no_grad():
    _, all_gate_logits = model(input_ids_viz)

# ── Plot 1: which expert each token uses in each MoE layer ──────────────────
fig, axes = plt.subplots(1, len(all_gate_logits), figsize=(5 * len(all_gate_logits), 4))
if len(all_gate_logits) == 1:
    axes = [axes]

for layer_i, (ax, gate_logits) in enumerate(zip(axes, all_gate_logits)):
    # gate_logits: (B*T, num_experts)
    probs = F.softmax(gate_logits, dim=-1)   # (B*T, num_experts)
    # Reshape to (B, T, num_experts) for visualization
    probs_2d = probs.view(batch_size_viz, seq_len_viz, config['num_experts'])
    # Show batch item 0
    sns.heatmap(
        probs_2d[0].numpy(),
        ax=ax, cmap='YlOrRd', vmin=0, vmax=0.5,
        xticklabels=[f'E{i}' for i in range(config['num_experts'])],
        yticklabels=[f'T{i}' for i in range(seq_len_viz)]
    )
    ax.set_title(f'MoE Layer {layer_i+1}\nGate probs (token × expert)')
    ax.set_xlabel('Expert')
    ax.set_ylabel('Token position')

plt.suptitle('Expert Routing Weights (brighter = higher probability)', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 2: Expert utilization histogram ─────────────────────────────────────
# How many tokens did each expert get assigned?

fig, axes = plt.subplots(1, len(all_gate_logits), figsize=(5 * len(all_gate_logits), 3))
if len(all_gate_logits) == 1:
    axes = [axes]

for layer_i, (ax, gate_logits) in enumerate(zip(axes, all_gate_logits)):
    _, top_experts = torch.topk(gate_logits, config['top_k'], dim=-1)  # (B*T, top_k)
    counts = [(top_experts == i).sum().item() for i in range(config['num_experts'])]
    ideal  = len(gate_logits) * config['top_k'] / config['num_experts']

    bars = ax.bar(range(config['num_experts']), counts,
                  color=['#E74C3C' if c > ideal*1.5 else '#2ECC71' for c in counts])
    ax.axhline(ideal, color='navy', linestyle='--', label=f'Ideal={ideal:.1f}')
    ax.set_title(f'MoE Layer {layer_i+1} Expert Load')
    ax.set_xlabel('Expert ID')
    ax.set_ylabel('Tokens assigned')
    ax.legend()

plt.suptitle('Expert Utilization (red = overloaded, green = balanced)', y=1.02)
plt.tight_layout()
plt.show()
print('In a well-trained model with aux loss, all bars should be close to the dashed line')

---
## 🧪 Part 10: Mini Training Loop

Let's do a small training run to see how the language modeling loss + auxiliary loss work together.

In [ ]:
import torch.optim as optim

# ── Dummy dataset ────────────────────────────────────────────────────────────
# In practice this would be real text tokenized into IDs
def get_batch(batch_size=4, seq_len=16, vocab_size=1000):
    x = torch.randint(0, vocab_size, (batch_size, seq_len))
    # Target: next token prediction (shift by 1)
    y = torch.randint(0, vocab_size, (batch_size, seq_len))
    return x, y

# ── Training setup ───────────────────────────────────────────────────────────
model_train = MiniDeepSeek(config)
optimizer   = optim.AdamW(model_train.parameters(), lr=1e-3)
aux_alpha   = 0.01  # weight for auxiliary load balancing loss

# ── Training loop ────────────────────────────────────────────────────────────
lm_losses, aux_losses, total_losses = [], [], []

for step in range(50):
    x, y = get_batch()
    optimizer.zero_grad()

    # Forward pass
    logits, all_gate_logits = model_train(x)
    # logits: (B, T, vocab_size)

    # ── Main language modeling loss (cross-entropy) ──────────────────
    # Reshape: (B*T, vocab_size) vs (B*T,)
    B, T, V = logits.shape
    lm_loss = F.cross_entropy(logits.view(B*T, V), y.view(B*T))

    # ── Auxiliary loss (sum across all MoE layers) ───────────────────
    aux_loss = torch.tensor(0.0)
    for gate_logits in all_gate_logits:
        _, top_experts = torch.topk(gate_logits, config['top_k'], dim=-1)
        aux_loss = aux_loss + auxiliary_load_balance_loss(
            gate_logits, top_experts,
            num_experts=config['num_experts'],
            alpha=aux_alpha
        )

    # ── Total loss ───────────────────────────────────────────────────
    total_loss = lm_loss + aux_loss
    total_loss.backward()
    optimizer.step()

    lm_losses.append(lm_loss.item())
    aux_losses.append(aux_loss.item())
    total_losses.append(total_loss.item())

    if (step + 1) % 10 == 0:
        print(f'Step {step+1:3d} | LM loss: {lm_loss.item():.4f} | Aux loss: {aux_loss.item():.4f} | Total: {total_loss.item():.4f}')

# ── Plot training curves ─────────────────────────────────────────────────────
plt.figure(figsize=(10, 4))
plt.plot(lm_losses,    label='LM loss (cross-entropy)',     linewidth=2)
plt.plot(aux_losses,   label='Aux loss (load balance)',     linewidth=2, linestyle='--')
plt.plot(total_losses, label='Total loss',                  linewidth=2, linestyle=':')
plt.xlabel('Training step')
plt.ylabel('Loss')
plt.title('Training Curves')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 📋 Summary: DeepSeek Architecture at a Glance

| Component | What it does | Key params |
|---|---|---|
| **Embeddings** | Token ID → dense vector | vocab=102400, dim=2048 |
| **RMSNorm** | Stabilizes activations | ε=1e-6 |
| **RoPE** | Injects position info by rotating Q,K | base=10000 |
| **Q,K,V projections** | Extract query/key/value from each token | 2048→2048 |
| **O projection** | Merge multi-head outputs | 2048→2048 |
| **gate_proj** | Learned soft filter (SiLU) | 2048→ffn_dim |
| **up_proj** | Content to be filtered | 2048→ffn_dim |
| **down_proj** | Project back to model dim | ffn_dim→2048 |
| **MoE Gate** | Route each token to top-K experts | 2048→64 experts |
| **Routed Experts** | 64 small specialized FFNs | inner dim=1408 |
| **Shared Expert** | 1 large always-on FFN | inner dim=2816 |
| **LM Head** | Predict next token probabilities | 2048→102400 |

### Key Design Choices
1. **Layer 0 is dense** — broad general processing before specialization begins
2. **Fine-grained experts** — more smaller experts (64×1408) vs fewer large ones
3. **Shared expert** — prevents routed experts from learning redundant common knowledge
4. **Aux loss** — keeps all experts active and specialized during training